# Apprendre une application solution

Ce notebook accompagne l'exercice 7.4 et prépare le chapitre sur la résolution d'équations. Pour

\[
u_t=-\mu u,\qquad u(0;\mu)=1,
\]

la solution exacte est $u(t;\mu)=e^{-\mu t}$. Nous apprenons la machine $(t,\mu)\mapsto u(t;\mu)$ à partir de valeurs de la solution.

## Parcours

1. [Données et domaines](#donnees-solution)
2. [Polynôme, SVR et MLP](#machines-solution)
3. [Interpolation et extrapolation](#erreurs-solution)
4. [Sensibilité au paramètre](#sensibilite-solution)
5. [Des données à l'équation](#equation-solution)

In [2]:
import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, scikit-learn 1.9.0


<a id="donnees-solution"></a>
## 1. Données et domaines

Construisez des données d'apprentissage et de validation pour $t\in[0,1]$ et $\mu\in[0,2.4]$. Le domaine $\mu\in(2.4,3]$ sera réservé à l'extrapolation. La fonction exacte permet ensuite d'évaluer les machines sur des grilles bien plus fines que les données.

In [ ]:
# À compléter.

<a id="machines-solution"></a>
## 2. Polynôme, SVR et MLP

Ajustez successivement une machine polynomiale, une SVR à noyau gaussien et un MLP. Les trois machines utilisent les mêmes valeurs de la solution, mais des classes de fonctions et des procédures d'apprentissage différentes.

In [4]:
class MLP(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.W1 = nnx.Linear(2, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.W3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.tanh(self.W1(x))
        x = nnx.tanh(self.W2(x))
        return self.W3(x).reshape(-1)


def perte_mlp(machine, x, z):
    return jnp.mean((machine(x) - z) ** 2)


@nnx.jit
def pas_mlp(machine, optimizer, x, z):
    valeur, gradient = nnx.value_and_grad(perte_mlp)(machine, x, z)
    optimizer.update(machine, gradient)
    return valeur


def entrainer_mlp(machine, x, z, *, alpha=2e-3, iterations=3000):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    x = jnp.asarray(x)
    z = jnp.asarray(z)
    historique = []
    for k in range(iterations):
        valeur = pas_mlp(machine, optimizer, x, z)
        if k % 100 == 0:
            historique.append(float(valeur))
    return historique

In [ ]:
# À compléter.

<a id="erreurs-solution"></a>
## 3. Interpolation et extrapolation

Calculez les erreurs $L^2$ et $L^\infty$ approchées sur les deux grilles. Représentez les solutions exactes et apprises dans le domaine d'interpolation, puis examinez séparément $\mu>2.4$.

In [ ]:
# À compléter.

<a id="sensibilite-solution"></a>
## 4. Sensibilité au paramètre

Utilisez la différentiation automatique pour calculer

\[

rac{\partial\Phi}{\partial\mu}(t,\mu)
\]

pour le MLP. Comparez-la à la sensibilité exacte $-t e^{-\mu t}$.

In [ ]:
# À compléter.

<a id="equation-solution"></a>
## 5. Des données à l'équation

La machine précédente ne voit que des couples $((t,\mu),u)$. Elle n'utilise ni l'équation $u_t+\mu u=0$ ni la condition initiale. Une méthode de type PINN ajoute au contraire des résidus de l'équation et des conditions aux limites à la fonction de perte. Le chapitre suivant examinera ce changement de cadre — et ses difficultés — sans supposer qu'un MLP résout automatiquement une équation différentielle.

## Bilan

Apprendre une application solution est déjà une réduction du coût d'évaluation : une fois entraînée, la machine fournit rapidement $u(t;\mu)$ et ses dérivées. Mais interpolation, extrapolation et sensibilité constituent trois critères distincts. La perte sur les données ne les contrôle pas automatiquement.